In dit notebook wordt uitgewerkt hoe je topics kunt verwijderen.

In [5]:
from ask_delphi_api.import_digicoach import Import

import_service = Import()

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!


In [6]:
def _delete_topic_relation(source_id: str, source_version_id: str, target_id: str, relation_type_id: str):
    """POST-call om een topic-relatie te verwijderen."""
    endpoint = (
        f"v2/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}"
        f"/topic/{source_id}/topicVersion/{source_version_id}/relation/delete"
    )
    data = {
        "relationTypeId": relation_type_id,
        "sourceTopicId": source_id,
        "targetTopicId": target_id,
    }
    return import_service.client._request("POST", endpoint, json_data=data)


def delete_relation(topic_id_source: str, topic_id_target: str, relation_name: str):
    """
    Verwijder een relatie van source → target van het type `relation_name`.
    Doet: checkout → resolve relation_type_id → delete → checkin → publiceer.
    """
    response = None
    try:
        import_service.topic.checkout(topic_id_source)
        source_version_id = import_service.topic.get_topicVersionId(topic_id_source)

        relation_type_id = import_service.relation.get_relation_type_id(
            topic_id_source, source_version_id, relation_name
        )

        response = _delete_topic_relation(
            topic_id_source, source_version_id, topic_id_target, relation_type_id
        )

    except Exception as e:
        print(f"Opruimen van topic relation mislukt: {e}")
    finally:
        # Deze twee moeten altijd gebeuren om de state netjes te houden
        try:
            import_service.topic.checkin(topic_id_source)
        finally:
            import_service.publiceer(topic_id_source)

    return response

def get_topic_relation(topicId: str):
    """Opvragen topic relations."""
    endpoint = f"v1/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topicId}/relation"
    data = {}
    return import_service.client._request("GET", endpoint, json_data=data)

def delete_topic(topic_id: str, version_id: str):
    """Voert de DELETE-call uit voor een topic."""
    endpoint = (
        f"v3/tenant/{{tenantId}}/project/{{projectId}}/"
        f"acl/{{aclEntryId}}/topic/{topic_id}/topicVersion/{version_id}"
    )
    data = {"workflowActions": {"increaseMajorVersionNo": True}}
    return import_service.client._request("DELETE", endpoint, json_data=data)


def soft_delete_topic(topic_id: str):
    """Soft delete: checkout → delete → checkin."""
    try:
        import_service.topic.checkout(topic_id)
        version_id = import_service.topic.get_topicVersionId(topic_id)
        response = delete_topic(topic_id, version_id)
    except Exception as e:
        print(f"Soft delete mislukt: {e}")
        response = {}
    finally:
        import_service.topic.checkin(topic_id)

    return response

def get_topic_ids(items: list, targetTopicType: str):
    topic_ids =[]
    for d in items:
        if d.get("targetTopicType") == targetTopicType and d.get("targetTopicIsDeleted") is False :
            topic_ids.append(d["targetTopicId"])
            print(f"{d['targetTopicName']}, {d["targetTopicId"]}")
    return topic_ids


### Verwijderen Digicoach taken, stappen en relaties

In [7]:
def delete_digicoach(topic_id_digicoach: str):

    response = get_topic_relation(topic_id_digicoach)

    # Ophalen topic ids taken
    task_topic_ids = get_topic_ids(response['relations'], "Task")
    print(f"task_topic_ids : {task_topic_ids}")

    for task_topic_id in task_topic_ids:

        # Ophalen topic ids van de stappen
        response = get_topic_relation(task_topic_id)
        action_topic_ids = get_topic_ids(response['relations'], "Action")
        print(f"action_topic_ids : {action_topic_ids}")

        # Verwijder stap topics en relations met de taken
        for action_topic_id in action_topic_ids:
            print(f"Verwijder stap topic {action_topic_id}")
            delete_relation(task_topic_id, action_topic_id, "Stap")
            soft_delete_topic(action_topic_id)

        # Verwijder taak topic en relation met de digicoach
        delete_relation(topic_id_digicoach, task_topic_id, "Taak")
        soft_delete_topic(task_topic_id)

    response = soft_delete_topic(topic_id_digicoach)
    print(response)

### Verwijder Digicoach proces

In [10]:
topic_id_digicoach = "77d3604a-db53-4499-964e-453e52c3c2fd"
# Digicoach Sanering (LIC) 2026-02-19 09:49:09.567104

delete_digicoach(topic_id_digicoach)

Selecteer Zaak, c8f3fd30-747b-439a-a391-0e02672d31ed
Quick scan verzoek, 1f8f0e51-d40d-4237-baa5-2f3372b73930
verzoek bekijken, 96a03662-aed4-46bc-b5fc-1269fd0d9c48
Verzoek beoordelen, e0da0404-c79c-4fbf-8af6-99a8a78f9751
Beschikking Uitbrengen, 6b11bb93-b457-45df-922f-3d778430dc72
task_topic_ids : ['c8f3fd30-747b-439a-a391-0e02672d31ed', '1f8f0e51-d40d-4237-baa5-2f3372b73930', '96a03662-aed4-46bc-b5fc-1269fd0d9c48', 'e0da0404-c79c-4fbf-8af6-99a8a78f9751', '6b11bb93-b457-45df-922f-3d778430dc72']
Open postbak in WAB, c324e92d-95f6-41b7-8359-b3e35e952829
Selecteer alleen kwijtscheldings / saneringsverzoeken, 6e7be17a-001a-4dad-867a-1248aa2c4749
In behandeling nemen poststuk saneringen, e8110ff2-a8a7-4c5c-87d9-3bbf3a82999d
Controleer taken op je naam, 5e43fd08-8eda-4683-a883-d1cadbac624b
action_topic_ids : ['c324e92d-95f6-41b7-8359-b3e35e952829', '6e7be17a-001a-4dad-867a-1248aa2c4749', 'e8110ff2-a8a7-4c5c-87d9-3bbf3a82999d', '5e43fd08-8eda-4683-a883-d1cadbac624b']
Verwijder stap topic c32